# Day 2: Data Cleaning & SQLite Load
**Bluestock Fintech — Mutual Fund Analytics Capstone**

Cleans NAV history, investor transactions, and scheme performance data,
then loads the cleaned fact tables plus `dim_fund` and `dim_date` into
`data/db/bluestock_mf.db`.

This notebook mirrors `scripts/etl_pipeline.py` — see that script for the
importable/reusable version used by `run_pipeline.py`.


In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
DB_PATH = Path("../data/db/bluestock_mf.db")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)


## Clean NAV history — reindex to full business-day range + forward-fill gaps

In [2]:
df_nav = pd.read_csv(RAW_DIR / "02_nav_history.csv")
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(['amfi_code', 'date']).drop_duplicates()

# Defensive: reindex each fund to a full business-day range and ffill any
# gaps. This does NOT insert weekend/holiday rows (which would distort
# volatility-based metrics downstream) — it only closes gaps a data feed
# might have missed on an actual trading day.
reindexed_parts = []
for code_, group in df_nav.groupby('amfi_code'):
    full_range = pd.bdate_range(group['date'].min(), group['date'].max())
    series = group.set_index('date')['nav'].reindex(full_range).ffill()
    part = series.reset_index().rename(columns={'index': 'date'})
    part['amfi_code'] = code_
    reindexed_parts.append(part[['amfi_code', 'date', 'nav']])

df_nav_clean = pd.concat(reindexed_parts, ignore_index=True)
df_nav_clean = df_nav_clean[df_nav_clean['nav'] > 0]

print(f"Rows before cleaning: {len(df_nav):,} | after: {len(df_nav_clean):,}")
df_nav_clean.to_csv(PROCESSED_DIR / "cleaned_nav_history.csv", index=False)


Rows before cleaning: 46,000 | after: 46,000


## Clean investor transactions

In [3]:
df_trans = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
df_trans['transaction_type'] = df_trans['transaction_type'].str.capitalize()
df_trans = df_trans[df_trans['amount_inr'] > 0]
df_trans['transaction_date'] = pd.to_datetime(df_trans['transaction_date'])

valid_kyc = ['Verified', 'Pending', 'Rejected']
df_trans = df_trans[df_trans['kyc_status'].isin(valid_kyc)]

print(f"Clean transaction rows: {len(df_trans):,}")
df_trans.to_csv(PROCESSED_DIR / "cleaned_investor_transactions.csv", index=False)


Clean transaction rows: 32,778


## Clean scheme performance

In [4]:
df_perf = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
df_perf['expense_ratio_pct'] = pd.to_numeric(df_perf['expense_ratio_pct'], errors='coerce')
df_perf = df_perf[(df_perf['expense_ratio_pct'] >= 0.1) & (df_perf['expense_ratio_pct'] <= 2.5)]

for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

print(f"Clean performance rows: {len(df_perf):,}")
df_perf.to_csv(PROCESSED_DIR / "cleaned_scheme_performance.csv", index=False)


Clean performance rows: 40


## Build dim_fund and dim_date, then load everything into SQLite

In [5]:
fund_master = pd.read_csv(RAW_DIR / "01_fund_master.csv")

dim_fund = fund_master[
    ['amfi_code', 'scheme_name', 'fund_house', 'category', 'sub_category', 'risk_category']
].rename(columns={'risk_category': 'risk_grade'}).drop_duplicates(subset=['amfi_code'])

all_dates = pd.concat([df_nav_clean['date'], df_trans['transaction_date']]).drop_duplicates().sort_values()
dim_date = pd.DataFrame({'date_id': all_dates.dt.strftime('%Y-%m-%d')})
dim_date['year'] = pd.to_datetime(dim_date['date_id']).dt.year
dim_date['month'] = pd.to_datetime(dim_date['date_id']).dt.month
dim_date['day'] = pd.to_datetime(dim_date['date_id']).dt.day
dim_date['quarter'] = pd.to_datetime(dim_date['date_id']).dt.quarter
dim_date['is_weekend'] = pd.to_datetime(dim_date['date_id']).dt.dayofweek >= 5
dim_date = dim_date.reset_index(drop=True)

engine = create_engine(f"sqlite:///{DB_PATH}")
tables = {
    'dim_fund': dim_fund,
    'dim_date': dim_date,
    'fact_nav': df_nav_clean,
    'fact_transactions': df_trans,
    'fact_performance': df_perf,
}
for name, df in tables.items():
    df.to_sql(name, engine, if_exists='replace', index=False)
    print(f"  loaded {name:20s} {df.shape[0]:>7,} rows")

print(f"\nDatabase written to {DB_PATH}")


  loaded dim_fund                  40 rows
  loaded dim_date               1,296 rows


  loaded fact_nav              46,000 rows


  loaded fact_transactions     32,778 rows
  loaded fact_performance          40 rows

Database written to ../data/db/bluestock_mf.db


## Verify the load

In [6]:
import sqlite3
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tables in DB:", [r[0] for r in cur.fetchall()])
conn.close()


Tables in DB: ['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance']
